# 2. Customer splits, complete models and comparisons

A small executable teaching example. Synthetic labels are deliberately encoded into fabricated curves; any AUROC below is a wiring check, never evidence of real-world accuracy. Prepare the pinned reference first with `python scripts/00_reference.py`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if ROOT.name != 'HEARTBEAT_LAB':
    ROOT = ROOT / 'HEARTBEAT_LAB'
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
from heartbeat_lab import prepare_days, allocate_customers, customer_cv

from heartbeat_lab.model import HeartbeatModel
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

## Create 40 fabricated customers

Each customer has seven days. The same person must stay together across all days and folds.

In [ ]:
rng = np.random.default_rng(20260930)
times = pd.date_range('2026-09-01', periods=7*288, freq='5min', tz='Europe/Berlin')
customers, records = [], []
for i in range(40):
    y = i % 2
    values = 70 + rng.normal(0, 4) + (6 + 3*y)*np.sin(np.arange(len(times))*2*np.pi/288 + .8*y)
    values += rng.normal(0, 2, len(times))
    frame = pd.DataFrame({'timestamp': times.astype(str), 'hr': values})
    frame = frame.loc[rng.random(len(frame)) > .15]
    days, hr, _ = prepare_days(frame)
    customers.append((days, hr))
    records.append({'customer_id': f'fake_{i:03d}', 'y': y})
customer_table = pd.DataFrame(records)
assignments = allocate_customers(customer_table)
assignments.groupby(['split','y']).size().unstack()


## Freeze the customer allocation

The real training script saves this manifest and reuses it for every option. These simple splits stratify the target; use the existing cohort_allocator for multivariable demographic balancing.

In [ ]:
membership = customer_table.merge(assignments[['customer_id','split']], on='customer_id', validate='one_to_one')
tr = np.flatnonzero(membership.split.eq('train'))
va = np.flatnonzero(membership.split.eq('val'))
te = np.flatnonzero(membership.split.eq('test'))
assert not set(tr) & set(va) and not set(tr) & set(te)

## Fit three complete pipelines

Summary ridge uses 305 B40-style features. MultiRocket adds pooled temporal features. Clock and global fills are fitted on training customers. Each alternative fits its own fill, kernels, feature imputer, scaler and ridge; no validation/test observations enter fit. Alpha=1000 is fixed. This may compile numba kernels on the first run.

In [ ]:
models = {
    'Summary ridge': HeartbeatModel(representation='summary'),
    'MultiRocket / clock fill': HeartbeatModel(filling='clock'),
    'MultiRocket / global fill': HeartbeatModel(filling='global'),
}
validation = []
for name, model in models.items():
    model.fit([customers[i] for i in tr], membership.iloc[tr].y.to_numpy())
    scores = model.predict([customers[i] for i in va])
    validation.append({'model': name, 'synthetic_validation_auc': roc_auc_score(membership.iloc[va].y, scores)})
pd.DataFrame(validation)


## Compare timelines for one held-out customer

Default: cumulative score using only days available by that date. Scores are not probabilities. Individual-day scores feed one day into a history-trained model and are exploratory. The same training seed, split and raw input make the two filling options comparable.

In [ ]:
days, hr = customers[te[0]]
fig, ax = plt.subplots(figsize=(10, 3))
for name, model in models.items():
    timeline = model.timeline(days, hr)
    ax.plot(timeline.date, timeline.cumulative_score, label=name)
    np.testing.assert_allclose(timeline.cumulative_score.iloc[-1], model.predict([(days, hr)])[0], rtol=1e-5)
ax.set(ylabel='Uncalibrated score toward salutation 20', xlabel='Date')
ax.legend()
fig.autofmt_xdate()
plt.show()


## Cross-validation for future tuning

Use only the outer training customers. The helper returns positional row indices, even when there are multiple day rows/customer. Fit a fresh entire pipeline in each fold. Do not globally fit MultiRocket and then CV only the ridge model.

In [ ]:
train_table = membership.iloc[tr][['customer_id','y']]
repeated_days = train_table.loc[train_table.index.repeat(7)].reset_index(drop=True)
fold_sizes = []
for train_rows, validation_rows in customer_cv(repeated_days, n_splits=3):
    train_ids = set(repeated_days.iloc[train_rows].customer_id)
    validation_ids = set(repeated_days.iloc[validation_rows].customer_id)
    assert train_ids.isdisjoint(validation_ids)
    fold_sizes.append((len(train_ids), len(validation_ids)))
fold_sizes


## Move to the real case study

Run numbered preparation/training scripts in the README. Keep all-days and first-40-day experiments separate. The historical 0.9264 endpoint had shared-transform exposure across allocations; this new pipeline requires its own evaluation. For app inference, each model lives beside metadata in artifacts/models/; never publish participant IDs or training data.